# 1g. Reductions and keepdim

About 20 minutes.

By the end you can write RMSNorm from its definition.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False, linewidth=110)


class _Todo:
    def __repr__(self):
        return "<fill me in>"


TODO = _Todo()


def S(t):
    """Shape as a tuple. Type this constantly."""
    return tuple(t.shape)


def check(label, got, want=None, shape=None, dtype=None, atol=1e-5, rtol=1e-4):
    """Check one answer. Prints what went wrong, then raises."""
    bad = []
    if isinstance(got, _Todo):
        bad.append("not filled in yet")
    else:
        if shape is not None:
            if not torch.is_tensor(got):
                bad.append(f"expected a tensor, got {type(got).__name__}")
            elif S(got) != tuple(shape):
                bad.append(f"shape is {S(got)}, want {tuple(shape)}")
        if dtype is not None and torch.is_tensor(got) and got.dtype != dtype:
            bad.append(f"dtype is {got.dtype}, want {dtype}")
        if want is not None:
            g = got if torch.is_tensor(got) else torch.as_tensor(got)
            w = want if torch.is_tensor(want) else torch.as_tensor(want)
            if S(g) != S(w):
                bad.append(f"shape is {S(g)}, want {S(w)}")
            elif not torch.allclose(g.double(), w.double(), atol=atol, rtol=rtol):
                bad.append(f"values differ, largest gap "
                           f"{(g.double() - w.double()).abs().max().item():.3e}")
    if bad:
        print(f"FAIL  {label}")
        for b in bad:
            print(f"      {b}")
        raise AssertionError(label)
    print(f"ok    {label}")


def same(label, a, b):
    """Two shapes or tuples must match."""
    if isinstance(a, _Todo):
        print(f"FAIL  {label}: not filled in yet")
        raise AssertionError(label)
    if tuple(a) != tuple(b):
        print(f"FAIL  {label}\n      you said {tuple(a)}, it is {tuple(b)}")
        raise AssertionError(label)
    print(f"ok    {label} = {tuple(a)}")

## dim says which axis to collapse

In [ ]:
x = torch.arange(12).float().reshape(3, 4)

print("x.sum()      ", S(x.sum()), "  everything -> rank 0")
print("x.sum(dim=0) ", S(x.sum(0)), " rows collapsed")
print("x.sum(dim=1) ", S(x.sum(1)), " columns collapsed")

`keepdim=True` leaves the axis behind as a length-1 blank axis.

In [ ]:
print("x.sum(-1)              ", S(x.sum(-1)))
print("x.sum(-1, keepdim=True)", S(x.sum(-1, keepdim=True)))

That is the whole purpose of `keepdim`: so the result **broadcasts back
against the input**. Every normalisation you write needs it.

In [ ]:
sq = torch.arange(16).float().reshape(4, 4) + 1

print("with keepdim, row sums:", (sq / sq.sum(-1, keepdim=True)).sum(-1).tolist())
print("without:               ",
      [round(f, 3) for f in (sq / sq.sum(-1)).sum(-1).tolist()])

The second line is wrong and raised nothing, because the tensor is square.
On a rectangular one you would have got an error.

## max and amax differ

`max(dim=...)` returns values **and** indices. `amax` returns only values.

In [ ]:
m = x.max(dim=-1)
print("x.max(-1) ->", type(m).__name__, m.values.tolist(), m.indices.tolist())
print("x.amax(-1) ->", x.amax(-1).tolist())

## Drill 1 (easy)

Compute `rms`: the root mean square of each row over the last axis, shaped so
that `x / rms` is correct.

In [ ]:
D = 8
x = torch.randn(3, D)

rms = TODO

check("rms", rms, shape=(3, 1))
check("x / rms has unit rows", (x / rms).pow(2).mean(-1), want=torch.ones(3))

## Drill 2 (medium) — RMSNorm

BioMistral's version, in words:

> Take `x`. In fp32, compute the reciprocal square root of the mean of its
> squares over the last axis, with `eps` added **to that mean**. Scale `x` by
> it. Round back to `x`'s dtype. *Then* multiply by `w`.

Three details, each a real divergence if you get it wrong:

1. `eps` goes inside the reciprocal square root, added to the mean.
2. Use the single-op reciprocal square root, not `1 / sqrt(...)`.
3. The cast back happens **before** the weight multiply.

No operation names are given. Retrieve them.

In [ ]:
def rms_norm(x, w, eps=1e-5):
    h = TODO            # fp32 copy
    h = TODO            # normalised
    return TODO         # cast back, then scale


xf = torch.randn(2, 3, D)
wf = torch.rand(D) + 0.5

got = rms_norm(xf, wf)
ref = wf * (xf / xf.pow(2).mean(-1, keepdim=True).add(1e-5).sqrt())

check("shape", got, shape=(2, 3, D))
check("matches an independent formulation", got, want=ref)
check("eps is inside, so zeros stay zero",
      rms_norm(torch.zeros(1, D), torch.ones(D)), want=torch.zeros(1, D))
check("bf16 in, bf16 out", rms_norm(xf.bfloat16(), wf.bfloat16()),
      dtype=torch.bfloat16, shape=(2, 3, D))

Detail 3 does nothing in fp32. In bf16 it changes the rounding, and that is
the kind of difference that shows up later as a logit mismatch. **1l** is
about why.

## Predict

In [ ]:
p = {"a": TODO, "b": TODO, "c": TODO, "d": TODO, "e": TODO}

z = torch.zeros(2, 3, 4, 5)
same("a  z.sum(1)", p["a"], S(z.sum(1)))
same("b  z.sum(2, keepdim=True)", p["b"], S(z.sum(2, keepdim=True)))
same("c  z.mean(0)", p["c"], S(z.mean(0)))
same("d  z.amax(-1, keepdim=True)", p["d"], S(z.amax(-1, keepdim=True)))
same("e  z.argmax(-1)", p["e"], S(z.argmax(-1)))

## Next

**1h** — matrix multiplication, and the weight layout every checkpoint uses.